In [ ]:
# ==========================================================================
# 생활용품 SNS 리뷰 감성 분석 웹 화면 (Gradio) — 셀 하나로 실행
# --------------------------------------------------------------------------
# [전체 흐름]
#   0. 도구 설치            : kiwipiepy, wordcloud, koreanize-matplotlib, gradio
#   1. 감성사전 준비        : KNU 한국어 감성사전 읽기 → 생활용품에 맞게 고치기
#   2. 분석 함수 만들기
#      2-1. 전처리          : 줄바꿈·가림 글자 정리 → 문장 나누기 → 인사·소개 문장 빼기
#      2-2. 단어 나누기     : Kiwi 형태소 분석 → 감성사전 모양의 단어 목록, 명사 목록
#      2-3. 감성 점수       : 단어 점수 합계 (안·못·않다·없다는 부호 뒤집기) → 긍정/중립/부정
#   3. 화면 동작 함수
#      3-1. analyze         : [분석하기] → 카테고리별 감성 비율 표·그래프, 워드클라우드, 대표 불만 문장
#      3-2. search          : [검색하기] → 키워드가 들어간 문장의 감성 비율(응답) + 문장 목록(검색 결과)
#   4. 웹 화면 실행         : 파일 선택 → 분석 결과 → 키워드 검색 순서로 화면 배치
#
# [사용 순서]
#   ① 이 셀 실행 → 셀 아래(또는 https://….gradio.live 주소)에 웹 화면이 나타남
#   ② 엑셀 파일 선택 → [분석하기] (30초~1분 정도 걸림)
#   ③ 키워드(예: 가격, 소음, 냄새)와 감성을 고르고 → [검색하기]
#   끝낼 때: 셀 왼쪽의 ■(중지) 버튼
#
# [엑셀 파일 조건] '카테고리', '상품평' 열이 있어야 함 (예: TS_5 생활용품 SNS 리뷰)
# [주의] 공개 주소(….gradio.live)는 주소를 아는 누구나 접속할 수 있음
# ==========================================================================


# ----- 0. 도구 설치 -----
# PIL(Pillow): Colab에 이미 설치된 이미지 도구, 지금 버전 번호를 알아내려고 불러옴
import PIL

# pip 설치 (pillow=={PIL.__version__}: 설치 중에 Pillow 버전이 바뀌어 ImportError가 나는 것을 막으려고 지금 버전으로 고정)
!pip install -q kiwipiepy wordcloud koreanize-matplotlib gradio pillow=={PIL.__version__}


# ----- 1. 감성사전 준비 -----
import os
import re
import csv
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib                    # 불러오기만 하면 그래프 한글 글꼴(나눔고딕)이 설정됨
from wordcloud import WordCloud
from kiwipiepy import Kiwi
import gradio as gr

# 그래프 축의 마이너스(-) 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

# 워드클라우드용 글꼴 파일 (koreanize_matplotlib에 들어 있는 나눔고딕)
FONT = os.path.join(os.path.dirname(koreanize_matplotlib.__file__), "fonts", "NanumGothic.ttf")

# 한국어 형태소 분석기
kiwi = Kiwi()

# KNU 감성사전 읽기: 한 줄에 '단어<탭>점수', 점수는 -2(매우 부정) ~ +2(매우 긍정)
url = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"
lex = pd.read_csv(url, sep="\t", header=None, names=["단어", "점수"], quoting=csv.QUOTE_NONE).dropna()

# {단어: 점수} 사전으로 만들기
knu = dict(zip(lex["단어"].str.strip(), lex["점수"].astype(int)))

# ✏️ 0점으로 바꿀 단어: 인사말(감사), 뜻이 다른 단어(용기=그릇), 감정과 관계없는 단어(많다, 받다, 크다),
#    제품이 없애는 대상(곰팡이, 먼지, 냄새 → "곰팡이가 싹 지워져요"는 칭찬)
for w in ["감사", "용기", "많다", "받다", "크다", "고르다", "곰팡이", "먼지", "냄새", "때", "얼룩", "부담", "걱정"]:
    knu[w] = 0

# ✏️ 추가하거나 점수를 바꿀 단어 (-2 ~ +2)
knu.update({"은은하다": 1, "풍성하다": 1, "별로": -2, "아쉽다": -1, "단점": -1, "흠": -1,
            "날카롭다": -1, "시끄럽다": -1, "찢어지다": -1, "거칠다": -1, "번거롭다": -1})


# ----- 2. 분석 함수 -----
# ✏️ 빼고 싶은 블로그 인사·소개 표현 (|는 '또는')
GREETING = "안녕|클릭|이웃|포스팅|댓글|구독|소개해|소개할|소개하려|오늘은|오늘도"

# ✏️ 그래프에서 뺄 흔한 명사
STOP = {"제품", "사용", "구매", "구입", "생각", "정도", "부분", "후기", "느낌", "처음", "이번", "가격대", "지금"}

# 부호를 뒤집는 말: 앞에 오는 말(안 좋다), 뒤에 오는 말(좋지 않다, 불편 없다)
FLIP_BEFORE = {"안", "못"}
FLIP_AFTER = {"않다", "없다"}

# 감성별 색 (긍정 파랑, 중립 회색, 부정 빨강)
COLORS = {"긍정": "#2a78d6", "중립": "#c3c2b7", "부정": "#e34948"}
ORDER = ["긍정", "중립", "부정"]


# 2-1. 전처리: 리뷰 표 → 문장 표 (리뷰ID, 카테고리, 문장)
def split_sentences(df):
    # 줄바꿈 → 띄어쓰기, 가림 글자(OO, O냄새의 O, ㅇㅇ, **) 지우기, 여러 칸 띄어쓰기 → 한 칸
    #   (?<![A-Za-z])O+(?![A-Za-z]) : 앞뒤에 다른 영어 글자가 없는 O (영어 단어 안의 O는 남김)
    text = (df["상품평"].astype(str)
            .str.replace(r"\n", " ", regex=True)
            .str.replace(r"(?<![A-Za-z])O+(?![A-Za-z])|ㅇ{2,}|\*{2,}", "", regex=True)
            .str.replace(r"\s+", " ", regex=True))

    # Kiwi로 리뷰마다 문장 나누기 → [리뷰 번호, 카테고리, 문장] 목록
    rows = []
    for i, sents in enumerate(kiwi.split_into_sents(text)):
        for s in sents:
            rows.append([i, df["카테고리"].iloc[i], s.text])
    sent = pd.DataFrame(rows, columns=["리뷰ID", "카테고리", "문장"])

    # 인사·소개 표현이 없고 10자 이상인 문장만 남기기
    keep = ~sent["문장"].str.contains(GREETING) & (sent["문장"].str.len() >= 10)
    return sent[keep].reset_index(drop=True)


# 2-2. 형태소 분석 결과 → 감성사전 모양의 단어 목록
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag.split("-")[0]           # 'VA-I' → 'VA' (뒤에 붙은 표시 떼기)
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")      # 동사·형용사: '다' 붙이기 (좋 → 좋다)
        elif tag == "XR":
            words.append(t.form + "하다")    # 어근: '하다' 붙이기 (은은 → 은은하다)
        elif tag in {"NNG", "NNP"} or t.form in {"안", "못", "별로"}:
            words.append(t.form)            # 명사, 부정·평가 표현은 그대로
    return words


# 2-3. 단어 목록 하나의 감성 점수
def score(words):
    total = 0
    for j, w in enumerate(words):
        if w in FLIP_BEFORE or w in FLIP_AFTER:   # 부정어 자체는 0점
            continue
        value = knu.get(w, 0)                     # 사전에 없으면 0점
        before = words[j - 1] if j > 0 else ""
        after = words[j + 1] if j + 1 < len(words) else ""
        if before in FLIP_BEFORE or after in FLIP_AFTER:
            value = -value                        # 앞뒤에 부정어가 있으면 부호 뒤집기
        total += value
    return total


# 점수 → 긍정(> 0) / 중립(= 0) / 부정(< 0)
def label(s):
    return "긍정" if s > 0 else ("부정" if s < 0 else "중립")


# ----- 3. 화면 동작 함수 -----
# 3-1. [분석하기] 버튼: 엑셀 → 문장 감성 → 표·그래프
def analyze(file_path):
    # 파일을 고르지 않았으면 안내 메시지만 돌려줌 (나머지 출력은 비움)
    if file_path is None:
        return "엑셀 파일을 먼저 선택하세요.", None, None, None, None, None

    # 엑셀 읽기, 필요한 열이 없으면 안내 메시지
    df = pd.read_excel(file_path).dropna(subset=["상품평"])
    if not {"카테고리", "상품평"} <= set(df.columns):
        return "'카테고리', '상품평' 열이 있는 엑셀 파일이 필요합니다.", None, None, None, None, None

    # 전처리 → 문장 표
    sent = split_sentences(df)

    # 모든 문장을 한 번에 형태소 분석 → 감성용 단어, 그래프용 명사(일반명사, 두 글자 이상)
    tokens = list(kiwi.tokenize(sent["문장"].tolist()))
    sent["단어"] = [to_words(t) for t in tokens]
    sent["명사"] = [[x.form for x in t if x.tag == "NNG" and len(x.form) > 1 and x.form not in STOP] for t in tokens]

    # 문장마다 감성 점수와 감성
    sent["점수"] = [score(w) for w in sent["단어"]]
    sent["감성"] = sent["점수"].apply(label)

    # 카테고리 × 감성 비율 표 (%) — 맨 아래에 전체 비율 한 줄 추가
    ratio = pd.crosstab(sent["카테고리"], sent["감성"], normalize="index")[ORDER] * 100
    ratio.loc["전체"] = sent["감성"].value_counts(normalize=True)[ORDER] * 100
    table = ratio.round(1).reset_index()

    # 그래프 1: 카테고리별 감성 비율 (가로 누적 막대, 칸 안에 % 표시)
    fig1, ax = plt.subplots(figsize=(9, 3.5))
    ratio.plot.barh(stacked=True, color=[COLORS[k] for k in ORDER], edgecolor="white", ax=ax)
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.0f%%", label_type="center", fontsize=9)
    ax.set_xlabel("문장 비율 (%)")
    ax.set_ylabel("")
    ax.invert_yaxis()                                   # '전체'가 맨 아래로 가도록 순서 뒤집기
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0), frameon=False)
    plt.tight_layout()
    plt.close(fig1)                                     # 셀 아래에 따로 그려지지 않게 닫기 (웹 화면에만 표시)

    # 그래프 2: 긍정 문장 vs 부정 문장 명사 워드클라우드 (글자가 클수록 많이 나온 명사)
    fig2, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, k in zip(axes, ["긍정", "부정"]):
        counts = Counter(n for nouns in sent[sent["감성"] == k]["명사"] for n in nouns)
        wc = WordCloud(font_path=FONT, background_color="white", width=600, height=400,
                       max_words=60, color_func=lambda *a, c=COLORS[k], **kw: c)
        ax.imshow(wc.generate_from_frequencies(counts))
        ax.set_title(f"{k} 문장의 명사")
        ax.axis("off")
    plt.tight_layout()
    plt.close(fig2)

    # 대표 불만 문장: 20~80자 부정 문장 중 점수가 낮은 순으로 카테고리마다 3개
    neg = sent[(sent["감성"] == "부정") & sent["문장"].str.len().between(20, 80)]
    worst = neg.sort_values("점수").groupby("카테고리").head(3).sort_values("카테고리")[["카테고리", "점수", "문장"]]

    # 처리 결과 메시지
    message = (f"리뷰 {len(df):,}건 → 문장 {len(sent):,}개 분석 완료. "
               f"긍정 {ratio.loc['전체', '긍정']:.0f}% · 중립 {ratio.loc['전체', '중립']:.0f}% · 부정 {ratio.loc['전체', '부정']:.0f}%")

    # 메시지, 비율 표, 그래프 2개, 대표 불만 문장, 보관함(검색에 쓸 문장 표)을 돌려줌
    return message, table, fig1, fig2, worst, sent[["카테고리", "문장", "점수", "감성"]]


# 3-2. [검색하기] 버튼: 키워드가 들어간 문장 → 응답(감성 비율) + 검색 결과(문장 목록)
def search(keyword, mood, sent):
    # 분석 전이거나 키워드가 비어 있으면 안내 메시지
    if sent is None:
        return "먼저 엑셀 파일을 올리고 [분석하기]를 누르세요.", None
    if not keyword.strip():
        return "키워드를 입력하세요. (예: 가격, 소음, 냄새, 손잡이)", None

    # 키워드가 들어간 문장 찾기 (regex=False: 키워드를 글자 그대로 찾음)
    found = sent[sent["문장"].str.contains(keyword.strip(), regex=False)]
    if len(found) == 0:
        return f"'{keyword}'이(가) 들어간 문장이 없습니다.", None

    # 응답: 키워드 문장의 감성 비율과 가장 부정적·긍정적인 문장
    pct = found["감성"].value_counts(normalize=True).reindex(ORDER, fill_value=0) * 100
    answer = (f"'{keyword}'이(가) 들어간 문장 {len(found):,}개: "
              f"긍정 {pct['긍정']:.0f}% · 중립 {pct['중립']:.0f}% · 부정 {pct['부정']:.0f}%\n\n"
              f"가장 아쉬운 문장: {found.loc[found['점수'].idxmin(), '문장']}\n"
              f"가장 좋은 문장: {found.loc[found['점수'].idxmax(), '문장']}")

    # 검색 결과: 고른 감성만 남기고, 부정이면 점수 낮은 순·그 외는 높은 순으로 30개
    if mood != "전체":
        found = found[found["감성"] == mood]
    result = found.sort_values("점수", ascending=(mood == "부정")).head(30)
    return answer, result


# ----- 4. 웹 화면 -----
# gr.Blocks(): 웹 화면 전체 틀, with 블록 안의 부품이 위에서부터 차례로 배치됨
with gr.Blocks() as demo:
    gr.Markdown("# 🧴 생활용품 리뷰 감성 분석")
    gr.Markdown("엑셀 파일을 올리면 리뷰를 **문장으로 나눠** 긍정·중립·부정을 매기고, 키워드로 아쉬운 점을 찾아 줍니다. "
                "(Kiwi 형태소 분석 + KNU 한국어 감성사전)")

    # 보이지 않는 보관함: analyze가 만든 문장 표를 담아 search에 넘겨줌
    data = gr.State()

    # ===== ① 파일 선택 =====
    gr.Markdown("## ① 엑셀 파일 선택")
    file = gr.File(label="리뷰 엑셀 파일 (.xlsx)", file_types=[".xlsx"])
    run_btn = gr.Button("분석하기", variant="primary")
    status = gr.Textbox(label="처리 결과")

    # ===== ② 분석 결과 =====
    gr.Markdown("## ② 분석 결과")
    table = gr.Dataframe(label="카테고리별 문장 감성 비율 (%)")
    plot1 = gr.Plot(label="카테고리별 감성 비율")
    plot2 = gr.Plot(label="워드클라우드")
    worst = gr.Dataframe(label="카테고리별 대표 불만 문장", wrap=True, column_widths=["18%", "8%", "74%"])

    # ===== ③ 키워드 검색 =====
    gr.Markdown("## ③ 키워드로 찾기")
    with gr.Row():
        keyword = gr.Textbox(label="키워드", placeholder="예) 가격, 소음, 냄새, 손잡이", scale=3)
        mood = gr.Radio(["전체", "긍정", "중립", "부정"], value="부정", label="보여 줄 감성", scale=2)
    # ✏️ 예시 키워드: 누르면 키워드 칸이 채워짐
    gr.Examples(examples=["가격", "소음", "손잡이", "향", "뚜껑"], inputs=keyword)
    search_btn = gr.Button("검색하기", variant="primary")
    answer = gr.Textbox(label="응답", lines=4)
    result = gr.Dataframe(label="검색 결과 (최대 30개)", wrap=True, column_widths=["18%", "62%", "8%", "12%"])

    # 버튼과 함수 연결: 함수가 돌려준 값이 outputs 부품에 순서대로 표시됨
    run_btn.click(analyze, inputs=file, outputs=[status, table, plot1, plot2, worst, data])
    search_btn.click(search, inputs=[keyword, mood, data], outputs=[answer, result])
    keyword.submit(search, inputs=[keyword, mood, data], outputs=[answer, result])   # Enter로도 검색

# 웹 화면 실행 (debug=True: 셀이 계속 실행되며 오류를 셀 아래에 보여 줌)
demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 9.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 92.2 MB/s eta 0:00:00
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3a5af2a3062556efa9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (http